# 01 · 一集 8 镜头闭环

前提：`00_setup.ipynb` 已通过，ComfyUI 在 8188 端口运行，`workflows/` 已导出 API 工作流。

顺序：资产（角色三视图 + 场景）→ 关键帧 → 草稿视频 → 配音 → 合成 → QC。
先 `--limit 2` 验证链路，再放全量；任何一步失败先停下来看日志。

In [12]:
import os, pathlib, json, shutil
WORK = pathlib.Path(os.environ.get("WORK", "/mnt/workspace/ai-comic-drama"))
os.chdir(WORK)
# Jupyter's kernel Python is the system one; make every following !python use the venv.
os.environ["VIRTUAL_ENV"] = str(WORK / "venv")
os.environ["PATH"] = str(WORK / "venv" / "bin") + os.pathsep + os.environ["PATH"]

# Profile defaults: cuda24g uses GGUF Q4 without Lightning LoRA.
profile = os.environ.get("MODEL_PROFILE", "mi300x")
if profile == "cuda24g":
    os.environ.setdefault("IMAGE_STEPS", "20")
    os.environ.setdefault("IMAGE_CFG", "4.0")
else:
    os.environ.setdefault("IMAGE_STEPS", "4")
    os.environ.setdefault("IMAGE_CFG", "1.0")
print("MODEL_PROFILE:", profile, "IMAGE_STEPS:", os.environ["IMAGE_STEPS"])

EP = pathlib.Path("episodes/ep01")
if not (EP / "episode.json").exists():
    shutil.copy2(EP / "episode.example.json", EP / "episode.json")
ep = json.loads((EP / "episode.json").read_text(encoding="utf-8"))
print(ep["title"], "shots:", len(ep["shots"]), "duration:", sum(s["duration"] for s in ep["shots"]), "s")
for s in ep["shots"]:
    print(s["id"], s["duration"], "s", s["shot_size"], s["camera"], "refs:", len(s.get("refs", [])))

MODEL_PROFILE: mi300x IMAGE_STEPS: 4
夜班电梯 shots: 8 duration: 40 s
s01 5 s medium static refs: 2
s02 5 s close-up slow push-in refs: 2
s03 5 s medium static refs: 2
s04 4 s detail slow push-in refs: 1
s05 5 s medium handheld refs: 2
s06 5 s close-up static refs: 2
s07 5 s wide static refs: 2
s08 6 s wide pull-out refs: 2


## 1. 角色三视图 + 场景主视觉

需要 `workflows/qwen_image_t2i.api.json`（Qwen-Image 20B T2I）。
如果没有下载 T2I 模型，也可以先用 `qwen_edit_2511_t2i.api.json` 的空白图兜底，
但 G3 必须人工确认脸和服装没有崩。

生成后**不要直接继续**：打开 `assets/characters/lin_sheet.png` 和两张场景图，
确认五要素一致；不满意就改提示词/Seed 重跑，直到满意为止。

In [13]:
!test -f workflows/qwen_image_t2i.api.json && python -m pipeline.run_batch --episode episodes/ep01/episode.json --stage sheets --workflow workflows/qwen_image_t2i.api.json || echo '缺少 workflows/qwen_image_t2i.api.json，先导出或改用 qwen_edit_2511_t2i.api.json'

device: cuda:0 NVIDIA A10 : cudaMallocAsync
workflow: /mnt/workspace/ai-comic-drama/workflows/qwen_image_t2i.api.json
episode : episodes/ep01/episode.json
[asset] lin -> /mnt/workspace/ai-comic-drama/assets/characters/lin_sheet.png
  ok /mnt/workspace/ai-comic-drama/assets/characters/lin_sheet.png  (342.5s)
[asset] loc_elevator -> /mnt/workspace/ai-comic-drama/assets/scenes/elevator_master.png
  ok /mnt/workspace/ai-comic-drama/assets/scenes/elevator_master.png  (30.2s)
[asset] loc_corridor -> /mnt/workspace/ai-comic-drama/assets/scenes/corridor_master.png
  ok /mnt/workspace/ai-comic-drama/assets/scenes/corridor_master.png  (18.2s)

stage=sheets failures=0 elapsed=391.0s


In [14]:
from PIL import Image
import pathlib
paths = [pathlib.Path("assets/characters/lin_sheet.png"),
         pathlib.Path("assets/scenes/elevator_master.png"),
         pathlib.Path("assets/scenes/corridor_master.png")]
for p in paths:
    print(p, "exists:", p.exists(), "size:", Image.open(p).size if p.exists() else None)

assets/characters/lin_sheet.png exists: True size: (1080, 1920)
assets/scenes/elevator_master.png exists: True size: (1080, 1920)


assets/scenes/corridor_master.png exists: True size: (1080, 1920)


## 2. 关键帧（先 2 个镜头）

Qwen-Image-Edit-2511，2 张参考图：角色三视图 + 场景主视觉。
先用 Lightning 4 步 / CFG 1.0 跑草稿；确认构图后再 40 步出正片。

如果 `REF2` 不存在，脚本会自动退化为单参考；但角色一致性会下降。

In [23]:
!python -m pipeline.run_batch --episode episodes/ep01/episode.json --stage images --workflow workflows/qwen_edit_2511_edit.api.json --limit 2 --steps ${IMAGE_STEPS:-4} --cfg ${IMAGE_CFG:-1.0}

device: cuda:0 NVIDIA A10 : cudaMallocAsync
workflow: /mnt/workspace/ai-comic-drama/workflows/qwen_edit_2511_edit.api.json
episode : episodes/ep01/episode.json
[image] s01 -> /mnt/workspace/ai-comic-drama/episodes/ep01/keyframes/s01_start.png  seed=101
  ok /mnt/workspace/ai-comic-drama/episodes/ep01/keyframes/s01_start.png  (230.4s)
[image] s02 -> /mnt/workspace/ai-comic-drama/episodes/ep01/keyframes/s02_start.png  seed=102
  ok /mnt/workspace/ai-comic-drama/episodes/ep01/keyframes/s02_start.png  (20.2s)

stage=images failures=0 elapsed=250.6s


In [16]:
from PIL import Image
import pathlib
paths = sorted((EP / "keyframes").glob("*_start.png"))[:2]
thumbs = []
for p in paths:
    im = Image.open(p).convert("RGB")
    im.thumbnail((360, 640))
    thumbs.append((p.name, im))
    print(p.name, im.size)
if thumbs:
    w = sum(im.width for _, im in thumbs) + 12 * (len(thumbs) - 1)
    h = max(im.height for _, im in thumbs)
    sheet = Image.new("RGB", (w, h), (30, 30, 30))
    x = 0
    for _, im in thumbs:
        sheet.paste(im, (x, 0))
        x += im.width + 12
    out = EP / "keyframes" / "contact_start.jpg"
    sheet.save(out, quality=88)
    print("contact sheet:", out)

人工检查：脸型、发型、瞳色、服装、配饰是否与三视图一致；有没有多手指/崩脸/文字。
不合格就调 `image_prompt` 或换 Seed，不要带着坏图往下跑。

## 3. 草稿视频（5B，先 2 个镜头）

5B 只验证运动是否成立，不追求画质。看：动作是否自然、有没有闪烁/人物复制、镜头是否稳定。

In [27]:
!python -m pipeline.run_batch --episode episodes/ep01/episode.json --stage videos --workflow workflows/wan22_5b_i2v.api.json --limit 2

device: cuda:0 NVIDIA A10 : cudaMallocAsync
workflow: /mnt/workspace/ai-comic-drama/workflows/wan22_5b_i2v.api.json
episode : episodes/ep01/episode.json
[video] s01 -> /mnt/workspace/ai-comic-drama/episodes/ep01/clips/s01.mp4  seed=101
  ok /mnt/workspace/ai-comic-drama/episodes/ep01/clips/s01.mp4  (758.9s)
[video] s02 -> /mnt/workspace/ai-comic-drama/episodes/ep01/clips/s02.mp4  seed=102
  ok /mnt/workspace/ai-comic-drama/episodes/ep01/clips/s02.mp4  (728.8s)

stage=videos failures=0 elapsed=1487.8s


In [28]:
!ls -lh episodes/ep01/clips
!for f in episodes/ep01/clips/*.mp4; do echo "== $f"; ffprobe -v error -show_entries format=duration -show_entries stream=width,height,r_frame_rate -of default=nw=1 "$f"; done

总计 1.1M
-rw-r--r-- 1 root root 547K 11月 12 16:54 s01.mp4
-rw-r--r-- 1 root root 489K 11月 12 17:06 s02.mp4


== episodes/ep01/clips/s01.mp4
width=960
height=1664
r_frame_rate=16/1
duration=5.063000
== episodes/ep01/clips/s02.mp4
width=960
height=1664
r_frame_rate=16/1
duration=5.063000


## 4. 配音

优先用 `assets/voices/lin.wav` + 准确的 `voice_text` 做 zero-shot。
没有参考音时，`--sft-speaker 中文女` 只是跑通链路的兜底，正式片请换成自己的音色。

In [30]:
!test -x venv-tts/bin/python && PYTHONPATH=$PWD/CosyVoice venv-tts/bin/python -m pipeline.tts_cosyvoice --episode episodes/ep01/episode.json --sft-speaker 中文女 || echo 'CosyVoice 未安装：先运行 scripts/05_install_cosyvoice.sh；本轮可只出字幕版'

CosyVoice 未安装：先运行 scripts/05_install_cosyvoice.sh；本轮可只出字幕版


In [20]:
!ls -lh episodes/ep01/audio | head -20

总计 0


## 5. 草稿合成

720×1280、veryfast、CRF 23，只用于看节奏和字幕位置。缺镜头会自动跳过并在 manifest 里记录。

In [21]:
!python -m pipeline.assemble --episode episodes/ep01/episode.json --draft

== normalize ==
WARN: missing clip episodes/ep01/clips/s01.mp4, skip
WARN: missing clip episodes/ep01/clips/s02.mp4, skip
WARN: missing clip episodes/ep01/clips/s03.mp4, skip
WARN: missing clip episodes/ep01/clips/s04.mp4, skip
WARN: missing clip episodes/ep01/clips/s05.mp4, skip
WARN: missing clip episodes/ep01/clips/s06.mp4, skip
WARN: missing clip episodes/ep01/clips/s07.mp4, skip
WARN: missing clip episodes/ep01/clips/s08.mp4, skip
no clips to assemble


In [22]:
import json, pathlib
m = json.loads(pathlib.Path("episodes/ep01/final/manifest.json").read_text(encoding="utf-8"))
print(json.dumps(m, ensure_ascii=False, indent=2))
print("draft:", m["output"])

FileNotFoundError: [Errno 2] No such file or directory: 'episodes/ep01/final/manifest.json'

## 6. 全量 8 镜头

G6 通过后再跑全量。批量前先确认：磁盘剩余 > 15GB、ComfyUI 日志没有 OOM/NaN、
每张关键帧都人工看过。

In [ ]:
!python -m pipeline.run_batch --episode episodes/ep01/episode.json --stage images --workflow workflows/qwen_edit_2511_edit.api.json --steps ${IMAGE_STEPS:-4} --cfg ${IMAGE_CFG:-1.0}
!python -m pipeline.run_batch --episode episodes/ep01/episode.json --stage videos --workflow workflows/wan22_5b_i2v.api.json

## 7. 14B 正片（可选，G7 之后再做）

先把 5B 的 8 个镜头全部看完，确认构图和运动都成立，再花额度跑 14B。
14B 只用 4-step LoRA 起步；20 步只给“值得”的镜头。

In [ ]:
# 先下载 14B（约 38GB），确认磁盘后再取消注释
# !python scripts/02_download_models.py --groups video-14b
# !python -m pipeline.run_batch --episode episodes/ep01/episode.json --stage videos --workflow workflows/wan22_14b_i2v.api.json --limit 1

## 8. 正片合成与 QC

1080×1920 / 30fps / H.264 / AAC 192k，字幕烧录，响度 `-14 LUFS`。
最终检查：时长、分辨率、音画同步、字幕不挡脸、角色一致性。

In [ ]:
# 有 BGM 时加 --bgm assets/bgm/theme.mp3
!python -m pipeline.assemble --episode episodes/ep01/episode.json
!ffprobe -v error -show_entries format=duration,size -show_entries stream=codec_name,width,height,r_frame_rate -of default=nw=1 episodes/ep01/final/ep01.mp4

## Gate 记录

- G3 首张关键帧耗时：____ 秒；合格率：____ / ____
- G4 5B 单镜头耗时：____ 秒；失败镜头：____
- G6 8 镜头闭环总耗时：____ 小时；成片时长：____ 秒
- G10 TTS 总耗时：____ 分钟；是否用参考音：____
- G11 成片路径：`episodes/ep01/final/ep01.mp4`

停机前运行 `bash scripts/99_stop_check.sh`，记录剩余额度，再手动点“关闭实例”。